# Ataque ALGEN sentence-level sobre embeddings DPED (SST-2)

Sequência de execução: (1) gerar uma vez as quatro tabelas DPED com ruído calibrado, (2) verificar a proveniência e executar o smoke test, (3) grade de ataque DPED e baseline `bert_pretrained`, (4) cópia dos resultados para o Drive. Smoke e pesquisa são salvos em pastas distintas. Cada tabela tem seu epsilon individual; a divulgação conjunta das quatro exige composição.

Plano: [plano-ataque-algen-dped.md](../docs/analises/plano-ataque-algen-dped.md). Implementação: [ataque-algen-sentence-level.md](../docs/implementação/ataque-algen-sentence-level.md).

Antes de executar: preencha `RUN_NAME` com o nome da run do ALGEN (pasta em `algen/SIH-ALGEN-main/outputs/google_flan-t5-small`, que deve conter `training_args_and_best_models.json`) e confirme que `DRIVE_CKPT` contém os arquivos `.pt` citados nesse JSON.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/otavio-hbf/DPED-Sistemas-Hibridos.git'
REPO = '/content/DPED-Sistemas-Hibridos'
BRANCH = 'robust-metodol'
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO], check=True)
subprocess.run(['git', '-C', REPO, 'status', '--short', '--branch'], check=True)

In [ ]:
%cd /content/DPED-Sistemas-Hibridos
!pip install -q -r requirements.txt evaluate rouge-score pot wandb

In [ ]:
import os
import subprocess
import sys
import shutil
from datetime import datetime

os.chdir(REPO)

# Nome da run do decodificador ALGEN (pasta dentro de algen/SIH-ALGEN-main/outputs/google_flan-t5-small)
RUN_NAME = 'yiyic_mmarco_english_maxlength32_train150000_batch_size128_lr0.0001_wd0.0001_epochs100'
REPO_CKPT = os.path.join(REPO, 'algen/SIH-ALGEN-main/outputs/google_flan-t5-small', RUN_NAME)
# Pasta no Drive com os arquivos .pt do decodificador
DRIVE_CKPT = '/content/drive/MyDrive/SIH/algen/checkpoint'

# O JSON do repositório guarda caminhos absolutos da máquina de treino.
# Gravo uma cópia no Drive com os caminhos apontando para DRIVE_CKPT.
import json
with open(os.path.join(REPO_CKPT, 'training_args_and_best_models.json')) as f:
    _data = json.load(f)
_data['best_models'] = [
    [*entry[:-1], os.path.join(DRIVE_CKPT, os.path.basename(entry[-1]))]
    for entry in _data['best_models']
]
with open(os.path.join(DRIVE_CKPT, 'training_args_and_best_models.json'), 'w') as f:
    json.dump(_data, f, indent=4)

ALGEN_CKPT = DRIVE_CKPT
DRIVE_RUNS = '/content/drive/MyDrive/SIH/privEmbeddings'
RUN_ID = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
DRIVE_RUN_DIR = os.path.join(DRIVE_RUNS, RUN_ID)
os.makedirs(DRIVE_RUN_DIR, exist_ok=True)

def persist_path(source, destination):
    if not os.path.exists(source):
        raise FileNotFoundError(f'Etapa não produziu o artefato esperado: {source}')
    target = os.path.join(DRIVE_RUN_DIR, destination)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if os.path.isdir(source):
        shutil.copytree(source, target, dirs_exist_ok=True)
    else:
        shutil.copy2(source, target)
    print('Salvo no Drive:', target)
    return target

def persist_embedding_exports(epsilons, stage):
    for epsilon in epsilons:
        tag = str(epsilon).replace('.', 'p')
        source = os.path.join('results/embeddings', f'glue_sst2_dp-distill_eps{tag}',
                              'dped_student_pre_downstream')
        persist_path(source, os.path.join('exports', stage, f'epsilon_{tag}'))

EPSILONS = ['8.0', '4.0', '2.0', '1.0']
TRAIN_SIZES = [1, 10, 30, 100, 300, 500, 1000]
# Ridge fixo até a decisão P6 (escolha por validação ou fixo)
RIDGE_VALUES = [1.0]
ATTACKER = 'algen/SIH-ALGEN-main/src/attacker_dped_sentence.py'

def export_dir_for(eps):
    tag = eps.replace('.', 'p')
    return f'results/embeddings/glue_sst2_dp-distill_eps{tag}/dped_student_pre_downstream'

EXPORT_EPS8 = export_dir_for('8.0')
print('ALGEN_CKPT exists:', os.path.isdir(ALGEN_CKPT))

## Etapa 1: gerar as tabelas DPED com ruído calibrado

A varredura treina um ensemble uma vez e gera os exports pré-downstream para ε em {1, 2, 4, 8}, usando o vocabulário completo fixo e sem DP-SGD downstream. O atacante usa todos os tokens exportados, exceto tokens especiais; não usa nem exporta `token_freq.npy`.

In [ ]:
subprocess.run([sys.executable, 'main.py', '--dataset', 'glue', '--task', 'sst2', '--embedding_privacy_sweep', '--delta', '1e-5', '--num_epochs', '5', '--batch_size', '16', '--max_train_samples', '16384', '--max_eval_samples', '4096', '--max_test_samples', '1024', '--subset_strategy', 'random', '--subset_seed', '42'], check=True)
persist_embedding_exports(EPSILONS, 'research')

In [ ]:
!ls {EXPORT_EPS8}

### Verificação de proveniência antes da grade

Carrega as quatro tabelas DPED e o BERT pelo mesmo caminho usado pelo atacante. Interrompe se os IDs não estiverem alinhados ou se duas matrizes forem idênticas. Execute novamente após modificar os exports ou a configuração.

In [ ]:
CONDITION_HASHES = None
sys.path.insert(0, os.path.dirname(ATTACKER))
import importlib
for module_name in ('attacker_dped_sentence', 'embedding_provenance', 'run_provenance'):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
attacker_module = importlib.import_module('attacker_dped_sentence')
preflight_conditions = getattr(attacker_module, 'preflight_conditions', None)
if not callable(preflight_conditions):
    revision = subprocess.check_output(['git', '-C', REPO, 'rev-parse', '--short', 'HEAD'], text=True).strip()
    raise RuntimeError(f'Código desatualizado em {ATTACKER} (commit {revision}); atualize o repositório e reexecute esta célula.')
if len(EPSILONS) != 4:
    raise ValueError('O preflight exige quatro condições de epsilon.')
conditions = preflight_conditions({eps: export_dir_for(eps) for eps in EPSILONS})
os.makedirs('results/attack', exist_ok=True)
with open('results/attack/preflight.json', 'w', encoding='utf-8') as f:
    json.dump(conditions, f, indent=2)
persist_path('results/attack/preflight.json', 'results_attack/preflight.json')
CONDITION_HASHES = conditions
for label, provenance in CONDITION_HASHES.items():
    print(label, provenance['table_array_sha256'])

## Etapa 2: smoke test do atacante

Poucos exemplos, para validar carregamento do checkpoint, export e escrita dos CSVs antes da grade completa.

In [ ]:
smoke_cmd = [sys.executable, ATTACKER, '--checkpoint_path', ALGEN_CKPT, '--export_dir', EXPORT_EPS8, '--source', 'dped', '--epsilon', '8.0', '--train_samples', '10', '--test_samples', '10', '--ridge', '1.0', '--run_kind', 'smoke', '--output_dir', 'results/attack_smoke/smoke']
smoke_result = subprocess.run(smoke_cmd, text=True, capture_output=True)
print('=== Smoke stdout ===\n' + (smoke_result.stdout or '(vazio)'))
print('=== Smoke stderr ===\n' + (smoke_result.stderr or '(vazio)'))
if smoke_result.returncode:
    raise RuntimeError(f'Smoke test terminou com código {smoke_result.returncode}; consulte o stderr acima.')
persist_path('results/attack_smoke/smoke', 'results_attack_smoke/smoke')

## Etapa 3: grade de ataque

- DPED: uma execução por ε × `train_samples` × ridge.
- Baseline `bert_pretrained`: lê as linhas do BERT para os mesmos IDs de tokens do export de ε=8.0. O baseline não depende de ε.

Os resultados vão para `results/attack/`. Cada execução gera `sentence_results.csv` e `summary.json`.

In [ ]:
if 'CONDITION_HASHES' not in globals() or CONDITION_HASHES is None:
    raise RuntimeError('Execute o preflight de proveniência antes da grade.')
if len(EPSILONS) != 4 or set(CONDITION_HASHES) != {f'dped:{eps}' for eps in EPSILONS} | {'bert_pretrained'}:
    raise RuntimeError('As condições mudaram; execute o preflight novamente.')

test_selection_hash = None
train_selection_hashes = {}

def run_attacker(args, condition):
    global test_selection_hash
    provenance = CONDITION_HASHES[condition]
    cmd = [sys.executable, ATTACKER, '--checkpoint_path', ALGEN_CKPT, '--run_kind', 'research',
           '--expected_table_sha256', provenance['table_array_sha256'],
           '--expected_token_ids_sha256', provenance['token_ids_array_sha256']] + [str(a) for a in args]
    if condition.startswith('dped:'):
        cmd.extend(['--expected_export_provenance_sha256', provenance['export_provenance_sha256']])
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)
    output_dir = args[args.index('--output_dir') + 1]
    relative = os.path.relpath(output_dir, 'results/attack')
    if relative == os.pardir or relative.startswith(os.pardir + os.sep):
        raise ValueError(f'Saída de pesquisa fora de results/attack: {output_dir}')
    persist_path(output_dir, os.path.join('results_attack', relative))
    with open(os.path.join(output_dir, 'summary.json'), encoding='utf-8') as f:
        summary = json.load(f)
    selection = summary['sentence_selection']
    test_hash = selection['test_selection_sha256']
    train_hash = selection['train_selection_sha256']
    train_key = (summary['train_samples'], summary['seed'])
    if test_selection_hash is not None and test_selection_hash != test_hash:
        raise ValueError(f'Seleção de teste divergente em {output_dir}; não agregue a grade.')
    if train_key in train_selection_hashes and train_selection_hashes[train_key] != train_hash:
        raise ValueError(f'Seleção de alinhamento divergente em {output_dir}; não agregue a grade.')
    test_selection_hash = test_hash
    train_selection_hashes[train_key] = train_hash

for eps in EPSILONS:
    tag = eps.replace('.', 'p')
    for n in TRAIN_SIZES:
        for r in RIDGE_VALUES:
            run_attacker(['--export_dir', export_dir_for(eps), '--source', 'dped', '--epsilon', eps,
                          '--train_samples', n, '--test_samples', 200, '--ridge', r,
                          '--output_dir', f'results/attack/dped_eps{tag}/n{n}_r{r}'], f'dped:{eps}')

for n in TRAIN_SIZES:
    for r in RIDGE_VALUES:
        run_attacker(['--export_dir', EXPORT_EPS8, '--source', 'bert_pretrained',
                      '--train_samples', n, '--test_samples', 200, '--ridge', r,
                      '--output_dir', f'results/attack/bert_pretrained/n{n}_r{r}'], 'bert_pretrained')

## Etapa 4: salvar resultados no Drive

In [ ]:
persist_path('results/attack', 'results_attack')
persist_embedding_exports(EPSILONS, 'research')
print('Todos os artefatos concluídos estão no Drive em:', DRIVE_RUN_DIR)